# Tutorial 1: Visualizing for neural activity during animal's running behavior

## Section 1: Exploring the data 

**LOADING PREPROCESSED DATA**


In the code below, we load the data file as a set of five different 1d and 2d arrays from the file 'data.npz': 

dff: standardized (z-scored) calcium activity (fluorescence signal/standard deviation) of all the neurons over time (size: [number of neurons , number of measurement points in time])

spikes: standardized (z-scored) neuronal firing rate (number of spikes/ std of spikes) of all the neurons over time (size: [number of neurons , number of measurement points in time])

times: time vector (in sec) (size: [number of measurement points in time])

position: position of the animal on the treadmill (cm) (size: number of measurement points in time])

velocity: movement velocity of the mouse on the treadmill (cm/s) (size: [number of measurement points in time])


In [ ]:
import sys
from pathlib import Path
import numpy as np
from matplotlib import pyplot as plt

# We now use the module 'pathlib' to read our data from the directory where this Python file is located
PROJECT_ROOT = Path(sys.executable).parents[2]
DATA_PATH = PROJECT_ROOT / "data/processed"


In [ ]:

with np.load(f"{DATA_PATH}/03_running_calcium_data.npz") as data:
    dff = data['dff'] # z-scored calcium activity of neurons(rows) over time(columns). each row is the activity of a single neuron 
    spikes = data['spikes'] # z-scored inferred spiking activity of neurons(rows) over time(columns).
    times = data['times'] # time in seconds of each data point (1-dimensional array)
    position = data['position'] # position of the mouse on the treadmill (1-dimensional array)
    velocity = data['velocity'] # the running speed of the mouse (1-dimensional array)


Exercise 1: 
1. How many neurons are there in the dataset? 
2. what is the first time point (value of first element in times array)? what is the second time point?
3. what is the time difference between two consequitive data points ? (e.g. times(2)-times(1))



The difference between time points is the 'period of acquistion'. You can use this to compute the data acquistion frequency (how many times per second do we have data) by simply calculating 1/period. Just plug in the value you computed for the period of acquisition above in the code snippet below to calculate the data acquisition frequency. How many data points per second do we have?

In [ ]:
# add the value of the period
period= 0.1
# np.median(np.diff(times))
# use the period to compute the sampling rate
sampling_ferquency=1/period

# print the result
print(sampling_ferquency)

In the code below, we plot the a activity of a single neuron (at the moment neuron #13) along with the running behavior of the mouse and its position on the treadmill. Run the code and check the graphs. What do we see in each graph?

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

neuron_id = 13 # index of the example neuron
time_range = (900, 1050) # time range to be plotted (in seconds), whole vector is to long to be clearly visualized

#getting the indices of data points that are within the time range given above
indices = np.logical_and(time_range[0]<times, times < time_range[1])

# calcium activity (1D array) of the neuron with the index neuron_id 
dff_n = dff[neuron_id]
# spiking activity (1D array) of the neuron with the index neuron_id
spikes_n = spikes[neuron_id]

# creating a figure of 10x8 cm with 4 rows of subplots 
fig, axes = plt.subplots(nrows=4, figsize=(10, 8))

# plotting calcium activity of the example neuron between time points given by the time range
axes[0].plot(times[indices], dff_n[indices], c='g', lw=0.5, label='dff')
axes[0].set(ylabel=' dff')

# plotting spiking activity of the example neuron between time points given by the time range
axes[1].plot(times[indices],spikes_n[indices], c='r', lw=0.5, label='spks')
axes[1].set(ylabel='deconvolved spikes')

# plotting the position of the mouse between time points given by the time range
axes[2].plot(times[indices],position[indices], c='k', lw=0.8, label='position')
axes[2].set(ylabel='Poisition (cm)')

# plotting the running speed of the mouse between time points given by the time range
axes[3].plot(times[indices],velocity[indices], c='b', lw=0.8, label='velocity')
axes[3].set(xlabel='time(s)', ylabel='speed (cm/s)')
for ax in axes:
    ax.legend(loc='upper left')

Exercise 2:
1. What is the maximum running speed of the animal?

TIP Q1: It is possible that arrays in your data contain NAN (Not A Number) values. 
To make sure you can work with arrays that contain NANs, numpy provides functions such as
'nanmin' (instead of min), 'nanmax' (instead of max), 'nanmean' (instead of mean) ...
these functions ignore NAN values in the computations.
 

2. What is the mean calcium activity (dff) of neuron #25 in between positions 10 - 50 cm? 

TIP Q2: look at the above example. There we chose indices between time points 900-1050. In this question we need to choose indices of positions that are between 10 and 50 cm. . 



Optional

3. what is the mean spike rate of neuron #14 when the animal is moving faster than 10cm/s?
4. How many laps did the animal complete on the treadmill?

# Section 2: Locomotion driven neural activities




How are behavioral changes reflected in neural responses? 

Do neurons respond similarly when the mouse is sitting still vs running ?

Here in this section we will explore how we can study associations of behavior (motor behavior) with neural responses. Specifically, we are interested if there are any specific patterns of neuronal activity around moments where the mouse starts running after standing still.

To explore this we will start by finding time points where the mouse starts running on the treadmill. The code below first takes a subset of the data in a time range that we are specifically interested in (between 150 and 210 seconds). Then, we create a new array that contains 0 if the animal is sitting still (speed < 0.1) and 1 if the animal is running (speed > 0.1). Finally, we create an array where we save the starting points of runs, so the moments where the previous array changes from 0 to 1. We also make a number of graphs for illustration.

Read through the code and make sure that you understand what the code is doing. Run the code and look at the graphs. What is represented by the red elements in each graph?

In [ ]:

from os import stat

# The code below first takes a subset of the data in a time range that 
# we are specifically interested in (between 150 and 210 seconds).

time_range = (150,210) # range of time to be plotted in seconds
#boolean array satisfying time range limits
indices = np.logical_and(time_range[0]<times, times < time_range[1])
# subsections of the velocity, time and calcium activity arrays
velocity_sub=velocity[indices]
times_sub=times[indices]
dff_n_sub=dff_n[indices]

# We create a new array that contains 0 if the animal is sitting still (speed < 0.1) 
# and 1 if the animal is running (speed > 0.1).
# boolean arrays of stationary and running periods (run threshold is set to 0.1 cm/s)
stationary=np.logical_and(velocity_sub>=0, velocity_sub<0.1) # mouse running speed <0.1 cm/s
running = (velocity_sub>0.1) # mouse running speed >0.1 cm/s

#plotting step by step the process finding the running-start time points
fig,axes = plt.subplots(3,1,figsize=(10, 6))

#Panel 1: velocity profile 
axes[0].plot(times_sub,velocity_sub, alpha=0.5, label='speed') # velocity vs time array in blue (alpha value indicates transperancy) 
axes[0].scatter(times_sub[stationary],velocity_sub[stationary],marker='o',c='k',s=2,label='stationary') # data points where the mouse is stationary are plotted as black dots
axes[0].scatter(times_sub[running], velocity_sub[running],marker='o',c='r',s=2,label='running') # data points where the mouse is running are plotted as red dots
axes[0].set(ylabel='speed (cm/s)') #label y axis

# creating a 1D array where stationary periods are 0 and running periods are 1
running_binary = np.ones_like(velocity_sub) #create an array of ones with the size of velocity array
running_binary[stationary]=np.zeros_like(running_binary[stationary]) # set stationary points to 0

# panel 2
axes[1].plot(times_sub,velocity_sub,alpha=0.5,label='speed') # plot velocity in blue
axes[1].scatter(times_sub,running_binary*10, marker='o', c='r',s=2,label='running binary') # plot in red the new binary array multiplied by 10 to make it visible 
axes[1].set(ylabel='speed (cm/s)') # add y label

# find running start points. 
# np.diff([x0 x1 x2 x3 x4]) returns an array of differences such that [x1-x0 x2-x1 x3-x2 x4-x3]
# np.argwhere(condition) returns the indeices of the array where the condition is satisfied. here it is where the difference equals to 1
run_start = np.argwhere(np.diff(running_binary)==1)
# Panel 3
axes[2].plot(times_sub,velocity_sub,alpha=0.5,label='speed') # velocity in blue
axes[2].scatter(times_sub[run_start],np.ones_like(run_start), marker='o', color='r',label='run start') # where running-binary switches from 0 to 1 in red.
axes[2].set(xlabel='time(s)',ylabel='speed (cm/s)') #add x-axis and y-axis label

# add legends that were specified in each plotting function with a label
for ax in axes:
            ax.legend(loc='upper left')


Using the idea above now we create a function that returns the indices of the timepoints where the mouse starts running. This function takes two arguments: an array that contains the speed of the mouse, and a single number that is the threshold speed above which we consider the mouse to be running (this is an optional argument that defaults to 0.1). 

The function first creates a new array that contains a conversion of the speed array to zeroes and ones for not running and running (using the threshold). Then it  creates another array that contains the indices where a 1 occurs after a 0 - you can do this in several different ways. Finally, the function should return that (last) array.

Read through the code. Do you understand what this function is doing?

In [ ]:
# function definition
# inputs: speed: running speed array
#         run_threshold: running speed below which we consider the animal stationary and above which we consider running
# outputs: run_start: an array of indices where run speed threshold is crossed
def find_run_start_indices( speed:np.ndarray,run_threshold=0.1):
    stationary = np.logical_and(speed>=0,speed<run_threshold)
    running = speed>run_threshold

    running_binary = np.ones_like(speed)
    running_binary[stationary]=np.zeros_like(running_binary[stationary])
    run_start = np.argwhere(np.diff(running_binary)==1)
    return run_start  

Exercise 3:
1. Using the function above, can you compute how many run start instances there are (how many times does the mouse start running)? 
2. How many run start instances are there if the running threshold is 1 cm/s instead of 0.1 cm/s?
3. Plot the run_speed between 300 - 350 seconds and mark the points where the mouse starts running with symbols in the graph. 
Use different markers for a treshold of 0.1 cm/s and a threshold of 1 cm/s, like in this piece of code using red and green markers (use code below):

```python
    plt.plot(times_sub,velocity_sub)
    plt.scatter(times_sub[run_start1],np.ones_like(run_start1), marker='^', color='r')
    plt.scatter(times_sub[run_start2],np.ones_like(run_start2), marker='o', color='g', alpha=0.5)"
```

Just visually looking at the data, which do you think is a better threshold? Or is there perhaps yet another threshold that you would consider better (try it first)? Why?

Now that we have a way to get all the indices where the mouse starts running for any threshold that we might give, we can start connecting this behavioural information to neural information.

The function below returns the response of a neuron within a given time window surrounding the run start instances. It takes a few different arguments:

* an array containing the indices of the run start, which is the outputof the function that we previously built (run_start_ind) 
* an array containing the calcium activity of the neurons over time (dff)
* the ID number of the specific neuron that we want to look at (neuron_id)
* the number of seconds BEFORE the runstart that we want to record the neural response (pre_sec, defaults to 1)
* the number of seconds AFTER the runstart that we want to record the neural response (post_sec, defaults to 4)

The output of the function is a 2D array of calcium activities over time in the defined window around each runstart.

Read the code and make sure you understand it.

In [ ]:
# function definition of "get_run_response_of_nid"
# inputs: run_start_ind: a 1D array of run_start indices
#         dff: calcium activity of all neurons
#         neuron_ind: index of the neuron of interest
#         pre_sec: time window before run start in seconds
#         post_sec: time window following run start in seconds
# outputs: run_response: 2D array where each row is the calcium activity of the neuron of interest around each run-start instance
def get_run_response_of_nid(run_start_ind: np.ndarray, dff: np.ndarray,neuron_ind: int, pre_sec = 1, post_sec=4):
    sampling_rate=30
    
    # number of data points that corresponds to the pre runstart window
    pre_window =pre_sec*sampling_rate
    # number of data points that corresponds to the post runstart window
    post_window = post_sec*sampling_rate

    # initialize the output 2D array of run_response with NANs
    n_r = len(run_start_ind)
    n_c = pre_window+post_window

    run_response = np.full((n_r,n_c),np.nan)
    
    # determining number of neurons(n_ofn) and the range of data points(n_times)
    n_ofn, n_times=np.shape(dff)

    # calcium activity of the neuron of interest
    dff_n = dff[neuron_ind]
    
    # response windows will cover data points between runstart(i)-prewindow to runstart(i)+post_window
    # as we don't want these indices to be out of the bounds (not negative and not larger than n_times) of the calcium activity array
    # we only consider runstart(i) instances where 
    # runstart(i) > pre_window 
    # runstart(i) < n_times- post_window 
    min_ind = pre_window # minimum rs index
    max_ind = n_times-post_window # maximum rs index

    # we loop over all run_start instances and add calcium response around each run_start 
    # to the run_response array
    i=-1
    for rs in run_start_ind:
        if np.logical_and(rs>min_ind, rs<max_ind):
            i=i+1
            run_response[i]=dff_n[np.arange(rs-pre_window,rs+post_window)]
         
    return run_response

Exercise 4: Plot average activity of neuron #14 around the point where the mouse starts running

To do this, you should first again create a 'run_start' array that contains all the indices where the mouse starts running. You can do this by calling the function you built for this above. Next, create an array that contains the calcium activity over time around these run start points. You can do that by using the function "get_run_response_of_nid" above. Now you have a 2D array that contains the calcium activity around the run start for each run start in the data. The script below includes these steps. 

Your task is to find the average response of the neuron over all the instances where the mouse starts running. 

Finally, the rest of the script plots this average (and the standard error) of the neural response over time to visualize your finding.

What can you conclude based on this graph?

In [ ]:

# Script to plot mean run response (and the error-bars plotted  around the mean of the run response).

# from a statistics package we import the function sem "standard error of the mean"
from scipy.stats import sem


run_start = find_run_start_indices(velocity,0.1)
print(len(run_start))
nid=14
#fig,ax = plt.subplots(1,1,figsize=(8, 4))
run_response_dff = get_run_response_of_nid(run_start,dff,nid)

#time points around the run response
t=np.arange(np.size(run_response_dff,1))/30
# compute mean run response for neuron 14 (nid=14)
# compute mean run response for neuron 14 (nid=14)
y_mean = np.nanmean(run_response_dff, axis=0)
# compute standard error of the mean of run response for  neuron 14
y_sem = sem(run_response_dff, axis=0)


plt.plot(t,y_mean)
# This plots a vertical line at the time of the actual run start in the graph.
plt.axvline(1,color='red',lw=1)
plt.fill_between(t,y_mean-y_sem,y_mean+y_sem,facecolor='blue', alpha=0.5)
plt.xlabel('time(s)')
plt.ylabel('dFF')
#ax[0].set_ylim(0.1,0.5)


OPTIONAL: Exercise 5
1. what is the percentage of cells for which their activity is at least 20% higher in between 1-2 seconds after the mouse starts running than in the second before it starts running? (in the plot of exercise 5: the post activity range is 2-3 seconds, pre-activity is between 0-1 seconds)

